# 02 Nested-CV 模型结果与不确定性

本模块只读取 `workspace/models/nested_cv_balanced` 的五折 out-of-fold (OOF) 预测。原始 TFLite 输出视为 logits，经 sigmoid 转为 0–1 分数；固定阈值为 probability ≥ 0.5。不会在 outer test 上搜索阈值作为正式结果。

In [1]:
from pathlib import Path
import os
import sys

def find_project_root(start=None):
    path = Path(start or Path.cwd()).resolve()
    for candidate in [path, *path.parents]:
        if (candidate / "project_config.py").is_file():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the project repository.")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from project_config import WORKSPACE_DIR

os.environ.setdefault("MPLBACKEND", "Agg")
os.environ.setdefault("MPLCONFIGDIR", str(WORKSPACE_DIR / "matplotlib_cache"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 120)

def require_columns(frame, required, name):
    missing = sorted(set(required) - set(frame.columns))
    if missing:
        raise ValueError(f"{name} is missing required columns: {missing}")

def save_figure(fig, stem):
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f"{stem}.png", dpi=300, bbox_inches="tight")
    fig.savefig(OUTPUT_DIR / f"{stem}.pdf", bbox_inches="tight")
    plt.close(fig)

def write_latex(frame, name, **kwargs):
    (OUTPUT_DIR / name).write_text(frame.to_latex(index=False, **kwargs), encoding="utf-8")

from scipy.special import expit
from sklearn.calibration import calibration_curve
from sklearn.metrics import (average_precision_score, brier_score_loss, confusion_matrix,
                             f1_score, precision_recall_curve, precision_score, recall_score,
                             roc_auc_score, roc_curve)
from project_config import BINARY_LABELS_PATH

PROBABILITY_THRESHOLD = 0.50
BOOTSTRAP_ITERATIONS = 5000
RANDOM_SEED = 2021
MODEL_ROOT = WORKSPACE_DIR / "models" / "nested_cv_balanced"
OUTPUT_DIR = WORKSPACE_DIR / "paper_analysis" / "02_model_results_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if MODEL_ROOT.name != "nested_cv_balanced":
    raise AssertionError("Formal analysis must use nested_cv_balanced only")
if not BINARY_LABELS_PATH.is_file():
    raise FileNotFoundError(BINARY_LABELS_PATH)
print(f"Formal model root: {MODEL_ROOT}")
print(f"Output directory: {OUTPUT_DIR}")

Formal model root: /path/to/strom_petrels/workspace/models/nested_cv_balanced
Output directory: /path/to/strom_petrels/workspace/paper_analysis/02_model_results_analysis


## 合并 OOF 预测并重算固定阈值指标

排序型指标使用连续 sigmoid 分数。Precision、recall、F1 与混淆矩阵全部使用预先规定的 0.5 概率阈值，因此不采用 outer test 上选择的 `optimal_threshold`。

In [2]:
parts = []
for fold_dir in sorted(MODEL_ROOT.glob("outer_*")):
    path = fold_dir / "predictions.csv"
    if path.is_file():
        part = pd.read_csv(path)
        require_columns(part, {"file", "true_label", "score"}, str(path))
        part["outer_fold"] = fold_dir.name.removeprefix("outer_")
        parts.append(part)
if len(parts) != 5:
    raise FileNotFoundError(f"Expected five completed folds under {MODEL_ROOT}; found {len(parts)}")

oof = pd.concat(parts, ignore_index=True)
oof["raw_logit"] = pd.to_numeric(oof["score"], errors="raise")
oof["probability"] = expit(oof["raw_logit"].to_numpy(float))
oof["true_label"] = pd.to_numeric(oof["true_label"], errors="raise").astype(int)
oof["predicted_label"] = (oof["probability"] >= PROBABILITY_THRESHOLD).astype(int)
oof["file_name"] = oof["file"].astype(str).map(lambda value: Path(value).name)

labels = pd.read_csv(BINARY_LABELS_PATH)
require_columns(labels, {"clip_id", "source_file", "negative_type", "species"}, "binary labels")
labels["file_name"] = labels["clip_id"].astype(str) + ".wav"
oof = oof.merge(labels[["file_name", "clip_id", "source_file", "negative_type", "species"]],
                on="file_name", how="left", validate="one_to_one")

if len(oof) != 2000 or oof["clip_id"].isna().any() or oof["clip_id"].nunique() != 2000:
    raise AssertionError("Formal OOF predictions must contain 2,000 uniquely matched clips")
if not oof["probability"].between(0, 1, inclusive="both").all():
    raise AssertionError("Sigmoid probabilities must be between 0 and 1")

def metric_row(frame):
    y = frame["true_label"].to_numpy(int)
    p = frame["probability"].to_numpy(float)
    pred = p >= PROBABILITY_THRESHOLD
    return {
        "samples": len(frame),
        "auroc": roc_auc_score(y, p),
        "auprc": average_precision_score(y, p),
        "precision": precision_score(y, pred, zero_division=0),
        "recall": recall_score(y, pred, zero_division=0),
        "f1": f1_score(y, pred, zero_division=0),
        "brier_score": brier_score_loss(y, p),
    }

overall_metrics = pd.DataFrame([metric_row(oof)])
fold_metrics = pd.DataFrame([
    {"outer_fold": fold, **metric_row(group)} for fold, group in oof.groupby("outer_fold", sort=True)
])
overall_metrics.to_csv(OUTPUT_DIR / "oof_metrics.csv", index=False)
fold_metrics.to_csv(OUTPUT_DIR / "fold_metrics.csv", index=False)
oof.to_csv(OUTPUT_DIR / "oof_predictions.csv", index=False)
display(overall_metrics)
display(fold_metrics)

,samples,auroc,auprc,precision,recall,f1,brier_score
0,2000,0.999932,0.999933,0.994018,0.997,0.995507,0.003727


,outer_fold,samples,auroc,auprc,precision,recall,f1,brier_score
0,01,438,1.000000,1.000000,0.990950,1.000000,0.995455,0.005489
1,02,360,1.000000,1.000000,1.000000,1.000000,1.000000,0.000367
2,03,372,1.000000,1.000000,1.000000,0.989247,0.994595,0.005233
3,04,438,1.000000,1.000000,0.995455,1.000000,0.997722,0.001510
4,05,392,0.999896,0.999897,0.984848,0.994898,0.989848,0.005892


## 录音级 clustered bootstrap

以 `source_file` 为抽样簇，在每次重复中有放回抽取与原始录音数相同的来源，并带入该来源的全部片段。这样保留同一录音内片段相关性。固定随机种子 2021，共 5,000 次。

In [3]:
rng = np.random.default_rng(RANDOM_SEED)
source_codes, sources = pd.factorize(oof["source_file"], sort=True)
y_array = oof["true_label"].to_numpy(int)
p_array = oof["probability"].to_numpy(float)
pred_array = p_array >= PROBABILITY_THRESHOLD
descending = np.argsort(-p_array, kind="mergesort")
sorted_y = y_array[descending]
sorted_p = p_array[descending]
threshold_endpoints = np.r_[np.flatnonzero(np.diff(sorted_p) != 0), len(sorted_p) - 1]

def weighted_metrics(cluster_multiplicity):
    weights = cluster_multiplicity[source_codes].astype(float)
    total_positive = np.sum(weights * y_array)
    total_negative = np.sum(weights * (1 - y_array))
    true_positive = np.sum(weights * y_array * pred_array)
    false_positive = np.sum(weights * (1 - y_array) * pred_array)
    false_negative = total_positive - true_positive
    precision_fixed = true_positive / (true_positive + false_positive) if true_positive + false_positive else 0.0
    recall_fixed = true_positive / total_positive if total_positive else 0.0
    f1_fixed = 2 * precision_fixed * recall_fixed / (precision_fixed + recall_fixed) if precision_fixed + recall_fixed else 0.0

    sorted_weights = weights[descending]
    cumulative_tp = np.cumsum(sorted_weights * sorted_y)[threshold_endpoints]
    cumulative_fp = np.cumsum(sorted_weights * (1 - sorted_y))[threshold_endpoints]
    curve_precision = cumulative_tp / np.maximum(cumulative_tp + cumulative_fp, 1)
    curve_recall = cumulative_tp / total_positive
    auprc = np.sum(np.diff(np.r_[0.0, curve_recall]) * curve_precision)
    tpr = np.r_[0.0, cumulative_tp / total_positive]
    fpr = np.r_[0.0, cumulative_fp / total_negative]
    auroc = np.trapezoid(tpr, fpr)
    return {"auroc": auroc, "auprc": auprc, "precision": precision_fixed,
            "recall": recall_fixed, "f1": f1_fixed}

cluster_draws = rng.multinomial(
    len(sources), np.repeat(1 / len(sources), len(sources)), size=BOOTSTRAP_ITERATIONS
)
bootstrap_rows = [weighted_metrics(draw) for draw in cluster_draws]

bootstrap = pd.DataFrame(bootstrap_rows)
point = metric_row(oof)
ci_rows = []
for metric in bootstrap.columns:
    low, high = np.quantile(bootstrap[metric], [0.025, 0.975])
    ci_rows.append({"metric": metric, "estimate": point[metric], "ci95_low": low, "ci95_high": high,
                    "bootstrap_unit": "source_file", "iterations": BOOTSTRAP_ITERATIONS,
                    "seed": RANDOM_SEED})
clustered_ci = pd.DataFrame(ci_rows)
clustered_ci.to_csv(OUTPUT_DIR / "clustered_bootstrap_ci.csv", index=False)
write_latex(clustered_ci, "model_metrics.tex", float_format="%.4f")
display(clustered_ci)

,metric,estimate,ci95_low,ci95_high,bootstrap_unit,iterations,seed
0,auroc,0.999932,0.999794,0.999995,source_file,5000,2021
1,auprc,0.999933,0.999797,0.999995,source_file,5000,2021
2,precision,0.994018,0.988695,0.998913,source_file,5000,2021
3,recall,0.997000,0.990539,1.000000,source_file,5000,2021
4,f1,0.995507,0.991534,0.998517,source_file,5000,2021


In [4]:
y = oof["true_label"].to_numpy(int)
p = oof["probability"].to_numpy(float)
precision_curve, recall_curve, _ = precision_recall_curve(y, p)
fpr, tpr, _ = roc_curve(y, p)

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
axes[0].plot(recall_curve, precision_curve, color="#d95f02", label=f"AUPRC = {point['auprc']:.3f}")
axes[0].axhline(y.mean(), color="gray", linestyle="--", label="Prevalence")
axes[0].set(title="Out-of-fold precision-recall curve", xlabel="Recall", ylabel="Precision", xlim=(0, 1), ylim=(0, 1.02))
axes[0].legend()
axes[1].plot(fpr, tpr, color="#4c78a8", label=f"AUROC = {point['auroc']:.3f}")
axes[1].plot([0, 1], [0, 1], color="gray", linestyle="--")
axes[1].set(title="Out-of-fold ROC curve", xlabel="False-positive rate", ylabel="True-positive rate", xlim=(0, 1), ylim=(0, 1.02))
axes[1].legend()
save_figure(fig, "pr_roc_curves")

fig, ax = plt.subplots(figsize=(9, 4.5))
fold_metrics.set_index("outer_fold")[["precision", "recall", "f1", "auprc", "auroc"]].plot.bar(ax=ax)
ax.set(title="Fixed-threshold and ranking metrics by outer fold", xlabel="Outer fold", ylabel="Score", ylim=(0, 1.02))
ax.tick_params(axis="x", rotation=0)
save_figure(fig, "fold_metrics")

In [5]:
fraction_positive, mean_predicted = calibration_curve(y, p, n_bins=10, strategy="quantile")
cm = confusion_matrix(y, p >= PROBABILITY_THRESHOLD, labels=[0, 1])

fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))
axes[0].hist(oof.loc[oof.true_label == 0, "probability"], bins=30, alpha=0.6, label="Negative", color="#6c757d")
axes[0].hist(oof.loc[oof.true_label == 1, "probability"], bins=30, alpha=0.6, label="Positive", color="#d95f02")
axes[0].axvline(PROBABILITY_THRESHOLD, color="black", linestyle="--")
axes[0].set(title="OOF score distribution", xlabel="Sigmoid model score", ylabel="Clips")
axes[0].legend()

axes[1].plot(mean_predicted, fraction_positive, marker="o", color="#4c78a8")
axes[1].plot([0, 1], [0, 1], color="gray", linestyle="--")
axes[1].set(title=f"Reliability diagram (Brier = {point['brier_score']:.3f})", xlabel="Mean predicted score", ylabel="Observed positive fraction", xlim=(0, 1), ylim=(0, 1))

image = axes[2].imshow(cm, cmap="Blues")
for (row, col), value in np.ndenumerate(cm):
    axes[2].text(col, row, f"{value}", ha="center", va="center")
axes[2].set(title="Confusion matrix at score ≥ 0.5", xlabel="Predicted class", ylabel="True class",
            xticks=[0, 1], yticks=[0, 1], xticklabels=["Negative", "Positive"], yticklabels=["Negative", "Positive"])
fig.colorbar(image, ax=axes[2], fraction=0.046)
save_figure(fig, "score_calibration_confusion")

print("Complete OOF analysis: 2,000 unique clips, sigmoid scores in [0, 1], balanced model only.")

Complete OOF analysis: 2,000 unique clips, sigmoid scores in [0, 1], balanced model only.
